In [0]:
from pyspark.sql.functions import to_timestamp, regexp_replace, trim
rawtable = spark.sql("SHOW TABLES IN stockmarket.raw")
listtables = rawtable.select("tableName").filter("tableName <> 'feedlinks' AND tableName <> 'timesofindia'")
display(listtables)


In [0]:
for row in listtables.collect():
    table_name = row['tableName']
    tan = spark.sql(f"SELECT * FROM stockmarket.raw.{table_name}")
    display(table_name)
    df_clean = tan.withColumn(
        "pubDate",
        to_timestamp(
            trim(regexp_replace("pubDate", "^[A-Za-z]+,\\s*", "")),  # strip "Mon, "
            "d MMM yyyy HH:mm:ss xx"
        )
    )
    #display(df_clean)
    cleandata = df_clean.dropDuplicates()
    cleandata.write.format("delta").mode("append").saveAsTable("stockmarket.sliver."+table_name)
    final=spark.sql("select * from stockmarket.sliver."+table_name)
    display(final)